# Rossmann Sales Forecasting - REST API Client Demo
Демонстрація взаємодії з ML-мікросервісом FastAPI виключно через REST API (requests).
Жоден ML-код напряму тут не імпортується.


In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = 'http://127.0.0.1:8000'
print(f'API Base URL: {BASE_URL}')


## 1. Перевірка працездатності мікросервісу (GET /health)


In [ ]:
health_res = requests.get(f'{BASE_URL}/health').json()
print('Service Health:', health_res)


## 2. Отримання метаданих датасету (GET /data/info)


In [ ]:
info_res = requests.get(f'{BASE_URL}/data/info').json()
print('Total stores:', info_res['total_stores'])
print('Candidate features:', info_res['all_candidate_features'])
print('Train range:', info_res['train_date_range'])
print('Test range:', info_res['test_date_range'])


## 3. Навчання моделей через REST API (POST /train)


In [ ]:
# Тренування LightGBM
lgb_payload = {
    'model_type': 'LightGBM',
    'model_name': 'LGBM_Notebook_Demo',
    'hyperparameters': {'n_estimators': 300, 'learning_rate': 0.1, 'max_depth': 5}
}
print('Training LightGBM model...')
resp_lgb = requests.post(f'{BASE_URL}/train', json=lgb_payload, timeout=120)
print('Status:', resp_lgb.status_code)
train_res = resp_lgb.json()
print('Training Time (s):', train_res['training_time_sec'])
print('Validation Metrics:', train_res['metrics']['validation'])
print('Test Metrics:', train_res['metrics']['test'])


## 4. Перегляд списку та деталей моделей (GET /models, GET /models/{name})


In [ ]:
models = requests.get(f'{BASE_URL}/models').json()
print(f'Saved models ({len(models)}):', [m['model_name'] for m in models])


## 5. Точковий та пакетний прогноз (POST /predict, POST /predict/batch)


In [ ]:
# Точковий прогноз для магазину 1 на відкритий та закритий день
pred_open = requests.post(f'{BASE_URL}/predict', json={
    'model_name': 'LGBM_Notebook_Demo',
    'record': {'Store': 1, 'Date': '2015-05-01', 'Open': 1, 'Promo': 1}
}).json()
print('Open day forecast:', pred_open['predictions'][0])

pred_closed = requests.post(f'{BASE_URL}/predict', json={
    'model_name': 'LGBM_Notebook_Demo',
    'record': {'Store': 1, 'Date': '2015-05-03', 'Open': 0, 'Promo': 0}
}).json()
print('Closed day forecast (must be 0):', pred_closed['predictions'][0])


## 6. Генерація Holdout та Scenario Forecast (POST /forecast)


In [ ]:
# Holdout forecast на тестовому періоді 2015 року
fc_resp = requests.post(f'{BASE_URL}/forecast', json={
    'model_name': 'LGBM_Notebook_Demo',
    'stores': [1, 2],
    'start_date': '2015-04-01',
    'end_date': '2015-05-01'
}).json()

fc_df = pd.DataFrame(fc_resp['forecasts'])
print(f'Retrieved {len(fc_df)} daily forecast rows.')
print('Aggregate test metrics:', fc_resp.get('aggregate_metrics'))
fc_df.head()
